In [1]:
# 构建简单的神经网络
%matplotlib inline
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
torch.__version__

'1.11.0+cu113'

In [3]:
# 决定用来训练的设备
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using {} device'.format(device))

Using cuda device


In [4]:
# 继承nn.Module实现模型
# 在__init__()方法初始化层
# 在forward()方法实现对输入的数据操作
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
            nn.ReLU()
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


In [5]:
# 将模型移至GPU
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
    (5): ReLU()
  )
)


In [6]:
# 将数据输入到模型，会自动执行forward()并做一些其他工作，不要直接调用forward()
# 使用softmax获取概率最大地类别
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
print(logits)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")


tensor([[0.0000, 0.0193, 0.0128, 0.0000, 0.0000, 0.0164, 0.0375, 0.0367, 0.0000,
         0.0789]], device='cuda:0', grad_fn=<ReluBackward0>)
Predicted class: tensor([9], device='cuda:0')


In [7]:
print(f"First Linear weights: {model.linear_relu_stack[0].weight} \n")
print(f"First Linear weights: {model.linear_relu_stack[0].bias} \n")

First Linear weights: Parameter containing:
tensor([[ 2.9538e-02, -4.7757e-03,  1.7119e-02,  ..., -1.3046e-02,
         -1.3052e-02, -6.6818e-03],
        [-4.2845e-03,  2.6408e-02,  1.8330e-02,  ..., -2.1228e-02,
          2.3447e-02, -3.2846e-05],
        [ 5.4475e-03, -3.0699e-02,  2.4820e-02,  ...,  2.7473e-02,
          2.4739e-02,  1.9986e-02],
        ...,
        [-3.0362e-02, -1.8384e-02,  2.9648e-02,  ..., -5.5846e-03,
         -3.2552e-02, -2.1716e-02],
        [-3.1084e-03, -2.9039e-02,  5.7209e-03,  ...,  3.3727e-02,
          1.3497e-02,  1.5743e-02],
        [-1.9929e-02, -3.4236e-03, -2.8287e-02,  ...,  3.2580e-02,
         -3.4018e-02,  1.7844e-02]], device='cuda:0', requires_grad=True) 

First Linear weights: Parameter containing:
tensor([ 1.4606e-02, -2.4099e-02, -9.8531e-03, -1.1277e-02,  2.7278e-02,
         7.8955e-04,  2.2795e-02, -2.7708e-03, -1.0328e-02, -8.1293e-03,
         7.8045e-03,  1.9131e-02, -2.0698e-02,  3.4294e-02, -8.0676e-03,
         2.7592e-02,  

In [8]:
# 准备一个小批量数据，查看每一层都做了什么
input_image = torch.rand(3, 28, 28)
print(input_image.size())

torch.Size([3, 28, 28])


In [9]:
# Flatten将2维数据转换为连续的一维数据
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


In [10]:
# Linear将输入数据做线性变换:weight * input + bias
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [11]:
# Relu activation function
print(f"Before ReLU: {hidden1} \n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.7029, -0.1677,  0.0629,  0.7389,  0.1224,  0.2102, -0.2132,  0.2660,
          0.5068, -0.6126, -0.1183, -0.0922, -0.5806,  0.3564,  0.5374,  0.3648,
          0.5366, -0.0419,  0.2455, -0.3592],
        [ 0.7021,  0.0680, -0.2329,  0.6542,  0.5068, -0.2903,  0.3095,  0.1968,
          0.4720, -0.3400, -0.0662, -0.2715, -0.4334,  0.3415,  0.1850,  0.4637,
          0.0976, -0.1499, -0.0458, -0.1738],
        [ 0.8529, -0.1506, -0.2411,  0.6806,  0.3653, -0.0236,  0.2175,  0.0552,
          0.3352, -0.0749,  0.0900, -0.0543, -0.4100,  0.4715,  0.4382,  0.2924,
          0.0610, -0.1296,  0.1200, -0.2490]], grad_fn=<AddmmBackward0>) 


After ReLU: tensor([[0.7029, 0.0000, 0.0629, 0.7389, 0.1224, 0.2102, 0.0000, 0.2660, 0.5068,
         0.0000, 0.0000, 0.0000, 0.0000, 0.3564, 0.5374, 0.3648, 0.5366, 0.0000,
         0.2455, 0.0000],
        [0.7021, 0.0680, 0.0000, 0.6542, 0.5068, 0.0000, 0.3095, 0.1968, 0.4720,
         0.0000, 0.0000, 0.0000, 0.0000, 0.3415, 0.1

In [13]:
# Sequential
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)

In [14]:
# nn.SoftMax
softmax = nn.Softmax(dim=1)
pred_probab=softmax(logits)

In [16]:
print("Model structure: ", model, "\n\n")
for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values: {param[:2]} \n")


Model structure:  NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
    (5): ReLU()
  )
) 


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values: tensor([[ 2.9538e-02, -4.7757e-03,  1.7119e-02,  ..., -1.3046e-02,
         -1.3052e-02, -6.6818e-03],
        [-4.2845e-03,  2.6408e-02,  1.8330e-02,  ..., -2.1228e-02,
          2.3447e-02, -3.2846e-05]], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values: tensor([ 0.0146, -0.0241], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values: tensor([[ 0.0273, -0.0347, -0.0047,  ...,  0.0236, -0.0235, -0.0146],
        [ 0.0002,  0.0098, -0.03